# ATMS et résolution de conflits en C#

**Navigation** · [Sommaire](README.md) · [Version Python](Argumentation-06b-ATMS-Conflits-Python.ipynb) · [JTMS](Argumentation-06-JTMS-CSharp.ipynb)

Un JTMS étiquette une conclusion dans un état de croyances. Un ATMS conserve plusieurs environnements d’hypothèses, chacun suffisant pour dériver une même conclusion. Vous étudierez le produit des environnements, les invalidations, les contradictions et les conditions bloquantes, avant de comparer cinq stratégies de résolution de conflits entre agents.

Les appels passent par Python.NET vers les modules réellement utilisés dans le tronc, _atms_core et _conflict_resolution, copies issues du commit EPITA 713927e1 et référencées dans [NOTICE-EPITA](argumentation_lib/NOTICE-EPITA). Les scénarios, appels, contrôles et lectures sont écrits en C#. Une installation CPython 3.13 et le noyau .NET Interactive suffisent à ce parcours.

In [1]:
#r "nuget: pythonnet,3.1.0"
#load "../../Config/PythonNetLoader.cs"
#nullable enable
using System;
using System.Collections.Generic;
using System.Linq;
using System.Text.Json;
using Python.Runtime;

static void RequireArgument(bool ok, string message)
{
    if (!ok)
        throw new InvalidOperationException(message);
}

// Le package local expose ses moteurs sans outil distant. Ajouter le dossier du carnet au chemin du CPython embarqué.
if (PythonNetLoader.Initialize(typeof(PythonEngine).Assembly, "json")is not null)
    PythonEngine.BeginAllowThreads();
using (Py.GIL())
{
    using var boot = Py.CreateScope();
    boot.Exec("import sys, os; sys.path.insert(0, os.getcwd())");
    using var package = Py.Import("argumentation_lib");
    using var version = package.GetAttr("__version__");
    Console.WriteLine($"argumentation_lib {version.As<string>()} via Python.NET 3.1.0, .NET {Environment.Version}.");
}

Installing Packages pythonnet

argumentation_lib 0.2.0 via Python.NET 3.1.0, .NET 10.0.14.


## 1. Hypothèses et étiquettes

Une hypothèse apporte son propre singleton comme preuve de dérivation. Une étiquette est un ensemble d’environnements. Au démarrage, le nœud spécial ⊥ existe déjà et porte une étiquette vide.

In [2]:
#nullable enable
sealed class EpitaEngine : IDisposable
{
    readonly PyModule scope;
    public EpitaEngine(string moduleName, string typeName)
    {
        using var gil = Py.GIL();
        scope = Py.CreateScope();
        using var module = Py.Import(moduleName);
        using var type = module.GetAttr(typeName);
        using var engine = type.Invoke();
        scope.Set("engine", engine);
        scope.Exec("import json");
    }

    // Les expressions sont écrites dans le carnet. Les arguments externes traversent une chaîne JSON distincte.
    void Bind(object? args)
    {
        using var input = new PyString(JsonSerializer.Serialize(args));
        scope.Set("arguments_json", input);
        scope.Exec("a = json.loads(arguments_json)");
    }

    public void Apply(string expression, object? args = null)
    {
        using var gil = Py.GIL();
        Bind(args);
        using var result = scope.Eval(expression);
    }

    public T Read<T>(string expression, object? args = null)
    {
        using var gil = Py.GIL();
        Bind(args);
        using var result = scope.Eval($"json.dumps({expression},sort_keys=True,ensure_ascii=False)");
        return JsonSerializer.Deserialize<T>(result.As<string>()) ?? throw new InvalidOperationException("Pas de valeur renvoyée par le module.");
    }

    public void Dispose()
    {
        using var gil = Py.GIL();
        scope.Dispose();
    }
}

sealed class Atms : IDisposable
{
    readonly EpitaEngine runtime = new("argumentation_lib._atms_core", "ATMS");
    public void Assume(string belief) => runtime.Apply("engine.add_assumption(a)", belief);
    public void Node(string name) => runtime.Apply("engine.add_node(a)", name);
    public void Rule(string[] yes, string[] no, string name) => runtime.Apply("engine.add_justification(a['Yes'],a['No'],a['Name'])", new { Yes = yes, No = no, Name = name });
    public string[] Nodes() => runtime.Read<string[]>("sorted(engine.nodes)");
    public string[] Assumptions() => runtime.Read<string[]>("sorted(engine.get_assumptions())");
    public string[][] Environments(string name) => runtime.Read<string[][]>("sorted(sorted(e) for e in engine.get_environments(a))", name);
    public void Invalidate(string[] environment) => runtime.Apply("engine.invalidate_environment(frozenset(a))", environment);
    public bool Consistent(string[] environment) => runtime.Read<bool>("engine.is_consistent(frozenset(a))", environment);
    public int RuleCount(string name) => runtime.Read<int>("len(engine.nodes[a].justifications)", name);
    public JsonElement Explain(string name) => runtime.Read<JsonElement>("engine.explain_node(a)", name);
    public void Dispose() => runtime.Dispose();
}

static string PrintEnvs(string[][] label) => "[" + string.Join(", ", label.Select(env => "{" + string.Join(", ", env) + "}")) + "]";
{
    using var model = new Atms();
    model.Assume("p");
    model.Assume("q");
    model.Node("r");
    model.Rule(["p", "q"], [], "r");
    RequireArgument(model.Environments("r").Single().SequenceEqual(new[] { "p", "q" }) && model.Nodes().Contains("⊥") && model.Environments("⊥").Length == 0, "La justification doit réunir p et q.");
    foreach (string node in new[]
    {
        "p",
        "q",
        "r"
    }

    )
        Console.WriteLine($"{node} → {PrintEnvs(model.Environments(node))}");
    Console.WriteLine($"Hypothèses [{string.Join(", ", model.Assumptions())}] | nœuds [{string.Join(", ", model.Nodes())}]");
}

p → [{p}]


q → [{q}]


r → [{p, q}]


Hypothèses [p, q] | nœuds [p, q, r, ⊥]


### Lecture de l’étiquette

Les hypothèses p et q sont indépendantes. La conclusion r demande les deux à la fois. Elle a un seul environnement, {p, q}, qui n’est ni p seul ni q seul.

## 2. Produit cartésien et union des hypothèses

Lorsque chaque prémisse porte plusieurs environnements, on combine un environnement de chacune. Chaque combinaison est réduite à l’union de ses hypothèses, puis les doublons exacts sont éliminés par la représentation en ensemble. Dans le moteur exposé ici, un environnement plus grand peut subsister lorsqu’une de ses sous-parties suffit déjà.

In [3]:
{
    using var product = new Atms();
    foreach (string premise in new[]
    {
        "p",
        "s",
        "q"
    }

    )
        product.Assume(premise);
    product.Node("z");
    product.Node("v");
    product.Node("w");
    product.Rule(["p"], [], "z");
    product.Rule(["p", "s"], [], "z");
    product.Rule(["q"], [], "v");
    product.Rule(["z", "v"], [], "w");
    string[][] labelW = product.Environments("w");
    // L'invariant porte sur les environnements du moteur réel, y compris le sur-ensemble conservé.
    RequireArgument(labelW.Length == 2 && labelW[0].SequenceEqual(new[] { "p", "q" }) && labelW[1].SequenceEqual(new[] { "p", "q", "s" }), "Deux combinaisons distinctes doivent être affichées.");
    foreach (string node in new[]
    {
        "z",
        "v",
        "w"
    }

    )
        Console.WriteLine($"{node} → {PrintEnvs(product.Environments(node))}");
}

z → [{p}, {p, s}]


v → [{q}]


w → [{p, q}, {p, q, s}]


### Lecture des combinaisons

La proposition z peut être inférée sous {p} ou {p, s}, tandis que v dépend de {q}. Les deux combinaisons arrivant à w transportent q. L’exemple distingue la représentation réellement observée d’une antichaîne qui éliminerait systématiquement tous les sur-ensembles.

### Exercice 1 — Prédire quatre environnements

Construisez deux prémisses qui ont chacune deux supports, puis une conclusion qui exige les deux. Reconstituez le produit avant de comparer à l’ATMS.

In [4]:
#nullable enable
{
    using var exerciseAtms = new Atms();
    foreach (string p in new[]
    {
        "g1",
        "g2",
        "d1",
        "d2"
    }

    )
        exerciseAtms.Assume(p);
    exerciseAtms.Node("gauche");
    exerciseAtms.Node("droite");
    exerciseAtms.Node("conclusion");
    exerciseAtms.Rule(["g1"], [], "gauche");
    exerciseAtms.Rule(["g1", "g2"], [], "gauche");
    exerciseAtms.Rule(["d1"], [], "droite");
    exerciseAtms.Rule(["d1", "d2"], [], "droite");
    exerciseAtms.Rule(["gauche", "droite"], [], "conclusion");
    static string[][]? ProposerProduit(string[][] gauche, string[][] droite)
    {
        // TODO étudiant : former une union pour toute paire d'environnements.
        return null;
    }

    string[][] gauche = exerciseAtms.Environments("gauche"), droite = exerciseAtms.Environments("droite");
    Console.WriteLine($"gauche {PrintEnvs(gauche)} | droite {PrintEnvs(droite)}");
    var réponseProduit = ProposerProduit(gauche, droite);
    if (réponseProduit is not null)
        RequireArgument(PrintEnvs(réponseProduit) == PrintEnvs(exerciseAtms.Environments("conclusion")), "Vérifiez l'ordre et les ensembles.");
    else
        Console.WriteLine("À compléter : proposer et comparer le produit.");
}

gauche [{g1}, {g1, g2}] | droite [{d1}, {d1, d2}]


À compléter : proposer et comparer le produit.


<null>

## 3. Invalider un environnement

L’invalidation retire l’environnement et les sur-ensembles des étiquettes. Elle laisse en place les justifications, ce qui permet de distinguer la perte d’un soutien d’un retrait de règle.

In [5]:
{
    using var invalidation = new Atms();
    invalidation.Assume("x");
    invalidation.Node("t");
    invalidation.Rule(["x"], [], "t");
    Console.WriteLine($"t avant : {PrintEnvs(invalidation.Environments("t"))}");
    invalidation.Invalidate(["x"]);
    // Observer séparément les étiquettes et la présence de la règle.
    RequireArgument(invalidation.Environments("t").Length == 0 && invalidation.Environments("x").Length == 0 && invalidation.RuleCount("t") == 1, "L'invalidation doit retirer les labels mais préserver une justification.");
    Console.WriteLine($"t après : {PrintEnvs(invalidation.Environments("t"))} | x après : {PrintEnvs(invalidation.Environments("x"))}.");
    Console.WriteLine($"Justifications de t : {invalidation.RuleCount("t")}.");
}

t avant : [{x}]


t après : [] | x après : [].


Justifications de t : 1.


### Lecture de l’invalidation

Le résultat révèle une distinction temporelle. Les étiquettes sont modifiées immédiatement, tandis que le graphe de justifications reste présent. Aucune redérivation n’est demandée dans ce scénario.

## 4. Une contradiction qui ne persiste pas dans le registre

On rend {m, n} dérivable pour ⊥, puis on regarde son étiquette et la réponse de is_consistent. Le moteur livré ne mémorise pas cette contradiction : il invalide l’environnement, y compris dans l’étiquette de ⊥. Cette limite est observable à l’interface.

In [6]:
{
    using var contradiction = new Atms();
    contradiction.Assume("m");
    contradiction.Assume("n");
    contradiction.Rule(["m"], [], "n");
    // La question sur le même ensemble est posée avant et après la justification de ⊥.
    bool consistentBefore = contradiction.Consistent(["m", "n"]);
    contradiction.Rule(["m", "n"], [], "⊥");
    bool consistentAfter = contradiction.Consistent(["m", "n"]);
    RequireArgument(consistentBefore && consistentAfter && contradiction.Environments("⊥").Length == 0, "Ce moteur ne conserve pas le nogood dans ⊥ après invalidation.");
    Console.WriteLine($"Cohérence {{m,n}} : avant {consistentBefore}, après {consistentAfter}");
    foreach (string node in new[]
    {
        "⊥",
        "n",
        "m"
    }

    )
        Console.WriteLine($"{node} → {PrintEnvs(contradiction.Environments(node))}");
}

Cohérence {m,n} : avant True, après True


⊥ → []


n → [{n}]


m → []


### Lecture de la limite

Un journal persistant des environnements incompatibles pourrait répondre différemment. Ici, observer vraie avant et après ne prouve pas que l’environnement n’a jamais été invalidé. Il faut regarder les étiquettes et connaître la durée de vie de l’information.

## 5. Bloquer une justification par inclusion

Un nœud OUT bloque une combinaison lorsqu’un de ses environnements est inclus dans l’environnement candidat. On compare deux cas. Dans le premier, w a besoin de u, alors que le candidat utilise s. Dans le second, s suffit aussi à w.

In [7]:
// Rejouer deux graphes qui ne diffèrent que par le soutien supplémentaire de w.
static Atms BlocksViaSubset(bool alsoS)
{
    var a = new Atms();
    a.Assume("s");
    a.Assume("u");
    foreach (string name in new[]
    {
        "v",
        "w",
        "bloque"
    }

    )
        a.Node(name);
    a.Rule(["s"], [], "v");
    if (alsoS)
        a.Rule(["s"], [], "w");
    a.Rule(["u"], [], "w");
    a.Rule(["v"], ["w"], "bloque");
    return a;
}

{
    using var unrelated = BlocksViaSubset(false);
    using var included = BlocksViaSubset(true);
    var allowedLabel = unrelated.Environments("bloque");
    var blockedLabel = included.Environments("bloque");
    RequireArgument(allowedLabel.Single().SequenceEqual(new[] { "s" }) && blockedLabel.Length == 0, "w bloque uniquement s'il est déjà dérivable sous le candidat.");
    Console.WriteLine($"w exige u : {PrintEnvs(allowedLabel)} | w existe sous s : {PrintEnvs(blockedLabel)}.");
    try
    {
        Console.WriteLine(PrintEnvs(unrelated.Environments("inconnu")));
    }
    catch (PythonException error)
    {
        RequireArgument(error.Type.Name == "KeyError", "Un nœud inconnu doit rester une erreur de clé du vrai moteur.");
        Console.WriteLine($"Nœud inconnu : {error.Type.Name}.");
    }
}

w exige u : [{s}] | w existe sous s : [].


Nœud inconnu : KeyError.


### Lecture des deux cas

Le blocage dépend du candidat, pas simplement de l’existence d’un environnement pour w. L’erreur sur le nœud inconnu permet au lecteur de distinguer un label vide d’un identifiant inexistant.

## 6. Expliquer une dérivation

On demande l’explication de la conclusion e et de l’hypothèse d. Le premier objet expose une justification. Le second désigne une hypothèse source, sans justification entrante.

In [8]:
{
    using var explaining = new Atms();
    explaining.Assume("d");
    explaining.Node("e");
    explaining.Rule(["d"], [], "e");
    // Afficher directement le document structuré produit par le moteur.
    foreach (string name in new[]
    {
        "e",
        "d"
    }

    )
        Console.WriteLine($"{name} : {explaining.Explain(name).GetRawText()}");
}

e : {"environments": [["d"]], "is_assumption": false, "justifications": [{"in_nodes": ["d"], "out_nodes": []}], "name": "e"}


d : {"environments": [["d"]], "is_assumption": true, "justifications": [], "name": "d"}


## 7. Cinq manières de trancher un conflit

La confiance choisit le score direct le plus élevé. La preuve pondère ce score par le nombre de justifications. Le consensus compare les votes de validité et demande au moins trois croyances. L’expertise dépend du domaine et du rôle présent. La chronologie ne décide à partir de dates que lorsqu’elles sont fournies. Nous soumettons les mêmes trois avis aux cinq stratégies.

In [9]:
var resolver = new EpitaEngine("argumentation_lib._conflict_resolution", "ConflictResolver");
var conflict = new
{
    conflict_id = "C-1",
    context = new
    {
        type = "hypothesis"
    },
    beliefs = new
    {
        sherlock = new
        {
            belief_name = "H1",
            confidence = .9,
            justification_count = 2
        },
        watson = new
        {
            belief_name = "H2",
            confidence = .7,
            justification_count = 3
        },
        informal = new
        {
            belief_name = "H3",
            confidence = .4,
            justification_count = 1
        }
    }
};
JsonElement Resolve(object item, string strategy) => resolver.Read<JsonElement>("engine.resolve(a['Item'],strategy=a['Strategy'])", new { Item = item, Strategy = strategy });
static string GetField(JsonElement result, string key) => result.TryGetProperty(key, out var value) && value.ValueKind != JsonValueKind.Null ? value.ToString() : "—";
static void ShowResolution(string label, JsonElement answer) => Console.WriteLine($"{label, -21} | résolu {GetField(answer, "resolved")} | agent {GetField(answer, "chosen_agent")} | {GetField(answer, "reasoning")}");
var strategies = resolver.Read<string[]>("engine.STRATEGIES");
RequireArgument(strategies.Length == 5, "Les cinq stratégies doivent venir du module.");
var responses = strategies.ToDictionary(strategy => strategy, strategy => Resolve(conflict, strategy));
foreach (var(strategy, response)in responses)
    ShowResolution(strategy, response);
// La confiance seule et la preuve pondérée n'élisent pas le même avis pour ce témoin.
RequireArgument(GetField(responses["confidence_based"], "chosen_agent") == "sherlock" && GetField(responses["evidence_based"], "chosen_agent") == "watson" && !responses["consensus"].GetProperty("resolved").GetBoolean(), "Les stratégies doivent réellement se départager.");

confidence_based      | résolu True | agent sherlock | Highest confidence: 0.90 by sherlock


evidence_based        | résolu True | agent watson | Best evidence score: 2.10 by watson


consensus             | résolu False | agent — | Consensus tie


agent_expertise       | résolu True | agent sherlock | Expert sherlock for domain hypothesis


temporal              | résolu True | agent sherlock | temporal not applicable (no belief carries a timestamp); fell back to confidence. Highest confidence: 0.90 by sherlock


### Lecture des stratégies

Le produit de la confiance par le nombre de justifications suffit ici à faire changer le gagnant. L’absence de votes explicites laisse le consensus sans vainqueur. Pour l’expertise et la chronologie, lisez aussi l’explication du moteur afin de savoir si la règle demandée s’est appliquée.

### Exercice 2 — Choisir entre confiance et nombre de preuves

Deux agents proposent A et B. La première croyance a confiance 0,8 et une justification. La deuxième a confiance 0,6 et quatre justifications. Prédisez quelle stratégie sélectionne Watson, puis exécutez la vraie résolution pour vérifier.

In [10]:
#nullable enable
var exConflict = new
{
    conflict_id = "C-EX2",
    context = new
    {
        type = "evidence"
    },
    beliefs = new
    {
        sherlock = new
        {
            belief_name = "A",
            confidence = .8,
            justification_count = 1
        },
        watson = new
        {
            belief_name = "B",
            confidence = .6,
            justification_count = 4
        }
    }
};
static string? NomDeLaStratégie(string[] options)
{
    // TODO étudiant : choisir celle qui compare des preuves pondérées.
    return null;
}

string? predictedStrategy = NomDeLaStratégie(strategies);
if (predictedStrategy is not null)
{
    // Un résolveur isolé ne change pas la série de mesures qui sera résumée à la fin.
    using var exerciseResolver = new EpitaEngine("argumentation_lib._conflict_resolution", "ConflictResolver");
    var answer = exerciseResolver.Read<JsonElement>("engine.resolve(a['Conflict'],strategy=a['Strategy'])", new { Conflict = exConflict, Strategy = predictedStrategy });
    RequireArgument(GetField(answer, "chosen_agent") == "watson", "Le scénario doit donner Watson.");
    ShowResolution(predictedStrategy, answer);
}
else
    Console.WriteLine("À compléter : prédire puis tester le nom de la stratégie.");

À compléter : prédire puis tester le nom de la stratégie.


## 8. Les entrées mal formées et la confiance absente

Une stratégie inconnue est une erreur d’appel. Un message sans dictionnaire de croyances est lui aussi rejeté. Lorsqu’une confiance manque, le moteur produit en revanche une réponse structurée non résolue, qui nomme l’agent concerné. Cette distinction permet à l’appelant de récupérer le bon type de défaut.

In [11]:
// Séparer les erreurs d’appel du résultat structuré rendu lorsque des données manquent.
foreach (var example in new (object Conflict, string Strategy)[]
{
    (conflict, "magie"),
    (new { conflict_id = "C-2" }, "confidence_based")
}

)
{
    try
    {
        ShowResolution(example.Strategy, Resolve(example.Conflict, example.Strategy));
    }
    catch (PythonException error)
    {
        RequireArgument(error.Type.Name == "ValueError", "Le moteur doit rejeter une entrée mal formée.");
        Console.WriteLine($"{example.Strategy} : {error.Type.Name} — {error.Message}");
    }
}

var missing = new
{
    conflict_id = "C-3",
    beliefs = new
    {
        sherlock = new
        {
            belief_name = "H1",
            confidence = .9
        },
        watson = new
        {
            belief_name = "H2"
        }
    }
};
var unresolved = Resolve(missing, "confidence_based");
var reports = unresolved.GetProperty("missing_confidence").EnumerateArray().Select(x => x.GetString()).ToArray();
RequireArgument(!unresolved.GetProperty("resolved").GetBoolean() && reports.SequenceEqual(new[] { "watson" }), "L'absence de confiance ne doit pas être imputée à zéro.");
ShowResolution("Confiance absente", unresolved);
Console.WriteLine("Agents sans confiance : " + string.Join(", ", reports));

magie : ValueError — Unknown conflict resolution strategy 'magie'; known: confidence_based, evidence_based, consensus, agent_expertise, temporal


confidence_based : ValueError — Conflict has no 'beliefs' mapping (agent -> belief dict); got keys ['conflict_id']


Confiance absente     | résolu False | agent — | No confidence for watson: the beliefs cannot be compared


Agents sans confiance : watson


### Lecture du contrat

Un nom de stratégie invalide signale un défaut de l’appelant. Une confiance absente signale une information manquante dans un conflit bien formé. Renoncer à trancher est ici un résultat explicite du moteur.

### Exercice 3 — Prédire la réponse en présence d’un champ absent

Sherlock indique X avec une confiance de 0,5. Watson indique Y sans confiance. Prédisez le drapeau de résolution et les agents à signaler, sans traiter la confiance absente comme une probabilité nulle.

In [12]:
#nullable enable
var missingExercise = new
{
    conflict_id = "C-EX3",
    beliefs = new
    {
        sherlock = new
        {
            belief_name = "X",
            confidence = .5
        },
        watson = new
        {
            belief_name = "Y"
        }
    }
};
static (bool Resolved, string[] Missing)? PrédireRésolutionSansConfiance()
{
    // TODO étudiant : appliquer le contrat à cette instance.
    return null;
}

var guessed = PrédireRésolutionSansConfiance();
if (guessed is { } g)
{
    using var checkResolver = new EpitaEngine("argumentation_lib._conflict_resolution", "ConflictResolver");
    var real = checkResolver.Read<JsonElement>("engine.resolve(a,strategy='confidence_based')", missingExercise);
    RequireArgument(g.Resolved == real.GetProperty("resolved").GetBoolean() && g.Missing.SequenceEqual(real.GetProperty("missing_confidence").EnumerateArray().Select(x => x.GetString())), "La prédiction doit correspondre au résultat structuré.");
}
else
    Console.WriteLine("À compléter : prédire le statut et les agents dont la confiance manque.");

À compléter : prédire le statut et les agents dont la confiance manque.


## 9. Les effets du contexte et du temps

Nous testons un consensus à deux agents, une expertise pour un domaine connu et son repli pour un domaine inconnu. Le dernier conflit oppose une croyance ancienne et plus confiante à une croyance récente. Le moteur doit choisir selon le critère effectivement demandé.

In [13]:
// Rejouer la même méthode avec un domaine absent pour observer le repli.
var twoAgents = new
{
    conflict_id = "C-4",
    beliefs = new
    {
        a = new
        {
            belief_name = "X",
            confidence = .9,
            valid = true
        },
        b = new
        {
            belief_name = "Y",
            confidence = .5,
            valid = false
        }
    }
};
var consensus = Resolve(twoAgents, "consensus");
var expert = Resolve(conflict, "agent_expertise");
var unknown = Resolve(new { conflict.conflict_id, conflict.beliefs, context = new { type = "inconnu" } }, "agent_expertise");
var withTime = new
{
    conflict_id = "C-5",
    beliefs = new
    {
        sherlock = new
        {
            belief_name = "H1",
            confidence = .3,
            timestamp = "2026-10-01T10:00:00"
        },
        watson = new
        {
            belief_name = "H2",
            confidence = .9,
            timestamp = "2026-10-28T09:00:00"
        }
    }
};
var chronological = Resolve(withTime, "temporal");
RequireArgument(!consensus.GetProperty("resolved").GetBoolean() && GetField(expert, "chosen_agent") == "sherlock" && GetField(unknown, "fallback_from") == "agent_expertise" && GetField(chronological, "chosen_agent") == "watson", "Le contexte et la chronologie doivent s'exercer dans les cas du carnet.");
ShowResolution("Consensus à deux", consensus);
ShowResolution("Expertise connue", expert);
ShowResolution("Expertise inconnue", unknown);
Console.WriteLine($"Repli depuis {GetField(unknown, "fallback_from")}, stratégie {GetField(unknown, "strategy_used")}.");
ShowResolution("Chronologie", chronological);

Consensus à deux      | résolu False | agent — | Consensus requires 3+ agents


Expertise connue      | résolu True | agent sherlock | Expert sherlock for domain hypothesis


Expertise inconnue    | résolu True | agent sherlock | agent_expertise not applicable (no expert mapped for 'inconnu'); fell back to confidence. Highest confidence: 0.90 by sherlock


Repli depuis agent_expertise, stratégie confidence_based.


Chronologie           | résolu True | agent watson | Most recent: watson at 2026-10-28T09:00:00


### Lecture du repli

Un résultat résolu sous un domaine inconnu ne prouve pas que l’expertise a trouvé un spécialiste. Le champ fallback_from dit quelle stratégie a renoncé. Pour une chronologie valide, le moteur regarde les dates renseignées.

## 10. Examiner le journal

Le résolveur conserve les résultats effectivement produits. Les erreurs d’appel, bloquées avant résolution, ne contribuent pas au journal. Les appels exploratoires des exercices utilisent une autre instance et ne changent pas ces compteurs.

In [14]:
var stats = resolver.Read<JsonElement>("engine.get_stats()");
int historyLength = resolver.Read<int>("len(engine.get_history())");
// Les quatre dernières résolutions s'ajoutent aux cinq stratégies et au cas sans confiance.
RequireArgument(historyLength == 10 && stats.GetProperty("total_conflicts").GetInt32() == historyLength && stats.GetProperty("resolved").GetInt32() + stats.GetProperty("unresolved").GetInt32() == historyLength, "Le journal doit réconcilier les décisions et les non-résolutions.");
Console.WriteLine($"Conflits {stats.GetProperty("total_conflicts")} | résolus {stats.GetProperty("resolved")} | non résolus {stats.GetProperty("unresolved")}.");
Console.WriteLine($"Stratégies {stats.GetProperty("by_strategy").GetRawText()} | journal {historyLength}.");
resolver.Dispose();

Conflits 10 | résolus 7 | non résolus 3.


Stratégies {"agent_expertise": 2, "confidence_based": 4, "consensus": 2, "evidence_based": 1, "temporal": 1} | journal 10.


## À retenir

- Une justification ATMS combine les environnements de ses prémisses puis applique les conditions OUT.
- Une invalidation agit sur les étiquettes. Le graphe de justifications et la mémoire des contradictions sont des mécanismes distincts.
- Un résultat de conflit contient une justification et éventuellement un repli. L’historique permet d’auditer les règles réellement appliquées.

## Vérifiez votre compréhension

**Pourquoi w bloque-t-il le candidat s seulement dans le second cas ?**

<details><summary>Réponse</summary>

Une condition OUT bloque lorsque w possède un environnement inclus dans le candidat. Dans l’exemple, w devient dérivable sous {s}, alors qu’un environnement exigeant u ne bloquerait pas ce candidat.

</details>

**Pourquoi une résolution par confiance peut-elle être enregistrée alors que l’expertise a été demandée ?**

<details><summary>Réponse</summary>

Si le domaine ne désigne aucun rôle applicable, le moteur se replie sur la confiance. Le champ fallback_from enregistre ce changement.

</details>

## Pour aller plus loin

- [JTMS et rétractations](Argumentation-06-JTMS-CSharp.ipynb).
- J. de Kleer, *An Assumption-based TMS*, Artificial Intelligence 28, 1986.
- [Version et provenance des modules locaux](argumentation_lib/NOTICE-EPITA).